# Japan Open World — one-cell Colab builder

Builds the **Japan Open World** Godot 4.6.3 project from source in this repository and produces a
playable **Linux x86_64** desktop build.

**What this notebook does**

1. Detects the Colab runtime and hardware.
2. Clones (or updates) the GitHub repository and records the exact commit SHA.
3. Installs only the system packages actually needed (`curl`, `unzip`, `rsync`).
4. Downloads the pinned **Godot 4.6.3** editor and matching export templates, verifying SHA-256.
5. Rebuilds the real-world data: Kyoto elevation (AWS Terrain Tiles) and OpenStreetMap roads/buildings.
6. Downloads CC0 PBR materials and an HDRI from Poly Haven, verifying sizes and hashes.
7. Imports assets, runs static validation, runs the runtime smoke test.
8. Exports the Linux build, smoke-tests the packaged binary, archives it and writes SHA-256 sums.
9. Optionally copies the build to Google Drive and publishes it as a GitHub Release.

**How to run**

- `Runtime → Run all` is enough. The single *RUN EVERYTHING* cell performs the whole build.
- To publish to Google Drive, you must authorise the Drive mount when prompted.
- To publish a GitHub Release, add a fine-grained token as a Colab Secret named `GITHUB_TOKEN`.

Nothing is hidden: every step prints its exact output path.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Environment + configuration
# ---------------------------------------------------------------------------
import os, sys, json, shutil, subprocess, textwrap, platform, time

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
print('Colab runtime :', IN_COLAB)
print('Python        :', platform.python_version())
print('Architecture  :', platform.machine())
print('CPUs          :', os.cpu_count())
try:
    import torch
    print('CUDA available:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
    print('NOTE: Godot rendering here is headless; a GPU does not accelerate the export step.')
except Exception:
    print('torch not present - CPU only runtime')

# ---- user configuration ---------------------------------------------------
REPO_URL   = 'https://github.com/Daniel-debug-boop/japan-game-openworld1.git'
REPO_REF   = 'main'          # branch, tag or commit SHA to build
WORKDIR    = '/content/japan_open_world' if IN_COLAB else os.path.abspath('.')
COPY_TO_DRIVE = True         # set False to skip the Drive step entirely
MAKE_RELEASE  = True         # requires a Colab Secret named GITHUB_TOKEN
RELEASE_TAG   = 'build-latest'

print('\nRepo   :', REPO_URL)
print('Ref    :', REPO_REF)
print('Workdir:', WORKDIR)


In [ ]:
# ---------------------------------------------------------------------------
# 2. Install system dependencies (idempotent)
# ---------------------------------------------------------------------------
def run(cmd, check=True, capture=False):
    print('\n$', cmd if isinstance(cmd, str) else ' '.join(cmd))
    res = subprocess.run(cmd, shell=isinstance(cmd, str), text=True,
                         capture_output=capture)
    if capture:
        if res.stdout: print(res.stdout[-4000:])
        if res.stderr: print(res.stderr[-2000:])
    if check and res.returncode != 0:
        raise SystemExit(f'command failed ({res.returncode})')
    return res

if IN_COLAB:
    run('apt-get -qq update')
    run('apt-get -qq install -y curl unzip rsync git python3-pip')
    run(f'{sys.executable} -m pip -q install --upgrade pillow numpy DracoPy')
else:
    run(f'{sys.executable} -m pip -q install --upgrade pillow numpy DracoPy')

for tool in ('curl', 'unzip', 'git'):
    print(f'{tool:8s}', shutil.which(tool))


In [ ]:
# ---------------------------------------------------------------------------
# 3. The single build cell - runs the entire pipeline
# ---------------------------------------------------------------------------
import os, subprocess, json, hashlib, time, pathlib

def sh(cmd, cwd=None, check=True):
    print(f'\n\033[36m$ {cmd}\033[0m')
    p = subprocess.run(cmd, shell=True, cwd=cwd)
    if check and p.returncode != 0:
        raise SystemExit(f'FAILED ({p.returncode}): {cmd}')
    return p.returncode

t0 = time.time()

# 3a. Clone or update the repository at the requested ref.
if os.path.isdir(os.path.join(WORKDIR, '.git')):
    sh(f'git fetch --all --tags --prune', cwd=WORKDIR)
    sh(f'git checkout {REPO_REF}', cwd=WORKDIR)
    sh(f'git reset --hard origin/{REPO_REF}' if REPO_REF == 'main' else 'git reset --hard', cwd=WORKDIR)
else:
    parent = os.path.dirname(WORKDIR) or '.'
    os.makedirs(parent, exist_ok=True)
    sh(f'git clone {REPO_URL} {WORKDIR}')
    sh(f'git checkout {REPO_REF}', cwd=WORKDIR)

commit_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=WORKDIR, text=True).strip()
print('Building commit:', commit_sha)

# 3b. Regenerate real-world data (cached files are reused automatically).
sh(f'{sys.executable} tools/build_world.py', cwd=WORKDIR)
sh(f'{sys.executable} tools/fetch_assets.py', cwd=WORKDIR)

# PLATEAU Kyoto 3D city models (DracoPy required; gitignored .pmesh payloads).
try:
    import DracoPy  # noqa: F401
except Exception:
    print('WARN: DracoPy not installed - skipping PLATEAU conversion')
else:
    sh(f'{sys.executable} tools/fetch_plateau.py', cwd=WORKDIR)

# 3c. Full verified build: Godot download -> import -> validate -> smoke -> export -> package.
sh('bash tools/build_release.sh', cwd=WORKDIR)

# 3d. Verify the artifacts actually exist and report hashes.
build_dir = os.path.join(WORKDIR, 'build_output')
expected = ['japan_open_world.x86_64', 'japan_open_world.pck',
            'japan_open_world-linux-x86_64.tar.gz', 'SHA256SUMS.txt', 'build_report.json']
missing = [f for f in expected if not os.path.exists(os.path.join(build_dir, f))]
if missing:
    raise SystemExit(f'missing build artifacts: {missing}')

print('\n=== artifacts ===')
for f in expected:
    p = os.path.join(build_dir, f)
    print(f'{os.path.getsize(p)/1e6:8.2f} MB  {p}')

print('\n=== SHA-256 ===')
print(pathlib.Path(build_dir, 'SHA256SUMS.txt').read_text())

print('\n=== build report ===')
print(pathlib.Path(build_dir, 'build_report.json').read_text())

print(f'\nTotal build time: {time.time() - t0:.1f}s')


In [ ]:
# ---------------------------------------------------------------------------
# 3b. Graphical visual acceptance on the real GPU  (Phase 4)
#
# This is the stage that answers the actual question: does the game render the
# Kyoto world? It deliberately does NOT pass --headless. It provisions a display,
# launches the *packaged* build with a real graphics backend, captures several
# views of the streamed world, and verifies the pixels independently.
#
# It reports honestly: when the runtime cannot create a graphics context it prints
# the driver output and marks the stage FAILED instead of claiming a pass.
# ---------------------------------------------------------------------------
import os, subprocess, glob

print('=== graphics capability ===')
subprocess.run('nvidia-smi || echo "no nvidia-smi on PATH"', shell=True)
icds = sorted(os.path.basename(p) for p in glob.glob('/usr/share/vulkan/icd.d/*'))
print('vulkan ICDs:', icds or 'NONE - Vulkan will not initialise here')
try:
    import torch
    print('torch:', torch.__version__,
          'cuda:', torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
except Exception as exc:
    print('torch unavailable:', exc)

# Prefer real Vulkan (Forward+) when an ICD exists, otherwise Mesa llvmpipe OpenGL.
RENDER_DRIVER = 'vulkan' if icds else 'opengl3'
print('chosen rendering driver:', RENDER_DRIVER)

# Best-effort: these are only needed for the graphical stage, so a failure here
# must not abort the build. setup_software_display.sh also works without root.
subprocess.run('apt-get -qq update 2>/dev/null; '
               'apt-get -qq install -y ffmpeg libgl1 libegl1 libvulkan1 '
               'mesa-vulkan-drivers xvfb libxkbcommon0 libfontconfig1 '
               'libxcursor1 libxi6 libxrandr2 libxinerama1 2>/dev/null || true',
               shell=True)

visual_env = os.environ.copy()
visual_env['RENDER_DRIVER'] = RENDER_DRIVER
visual_env['VIDEO_FRAMES'] = os.environ.get('VIDEO_FRAMES', '150')
visual_env['RESOLUTION'] = '1280x720'

visual = subprocess.run('bash tools/visual_acceptance.sh build_output/visual_diag',
                        shell=True, cwd=WORKDIR, env=visual_env)
visual_ok = visual.returncode == 0
print('\n=== VISUAL ACCEPTANCE:', 'PASS' if visual_ok else 'FAIL', '===')
if not visual_ok:
    print('The graphics stage failed. Read build_output/visual_diag/capture.log and '
          'build_output/logs for the driver error before assuming the GPU is at fault.')
    print('On a Tesla T4, ensure a Vulkan ICD is installed (mesa-vulkan-drivers or the '
          'NVIDIA driver) - otherwise Godot falls back to Mesa llvmpipe OpenGL.')


In [ ]:
# ---------------------------------------------------------------------------
# 4. Optional: publish to Google Drive
# ---------------------------------------------------------------------------
drive_path = None
if COPY_TO_DRIVE and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        dest_dir = '/content/drive/MyDrive/JapanOpenWorldBuilds'
        os.makedirs(dest_dir, exist_ok=True)
        stamp = time.strftime('%Y%m%d-%H%M%S')
        dest = os.path.join(dest_dir, f'japan_open_world-{stamp}')
        sh(f'rsync -a --info=stats1 {build_dir}/ "{dest}/"')
        drive_path = dest
        print('\nDrive copy:', drive_path)
    except Exception as exc:
        print('Drive copy skipped:', exc)
else:
    print('Drive copy not requested / not running in Colab.')


In [ ]:
# ---------------------------------------------------------------------------
# 5. Optional: publish artifacts as a GitHub Release
# Requires a fine-grained token with `contents: write`, stored as Colab Secret GITHUB_TOKEN.
# The token is read from Secrets and never printed.
# ---------------------------------------------------------------------------
release_url = None
if MAKE_RELEASE:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = os.environ.get('GITHUB_TOKEN')

    if not token:
        print('No GITHUB_TOKEN secret found - skipping release upload.')
        print('Add it via Colab: Tools -> Secrets -> GITHUB_TOKEN (contents:write).')
    else:
        import json as _json, urllib.request as _req
        api = 'https://api.github.com'
        headers = {'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json'}

        def gh(method, path, body=None, raw_path=None):
            url = api + path
            data = None if body is None else _json.dumps(body).encode()
            r = _req.Request(url, data=data, method=method, headers=headers)
            if raw_path:
                r.add_header('Content-Type', 'application/octet-stream')
                data = open(raw_path, 'rb').read()
                r = _req.Request(url, data=data, method=method, headers=headers)
            try:
                with _req.urlopen(r, timeout=300) as resp:
                    return _json.loads(resp.read().decode())
            except Exception as exc:
                print('GitHub API error:', exc)
                return None

        repo_path = REPO_URL.split('github.com/')[-1].removesuffix('.git')
        rel = gh('POST', f'/repos/{repo_path}/releases', {
            'tag_name': RELEASE_TAG,
            'name': f'Latest automated build ({commit_sha[:8]})',
            'body': f'Automated Colab build of commit {commit_sha}.',
            'draft': False, 'prerelease': True,
        })
        if rel and 'upload_url' in rel:
            upload = rel['upload_url'].split('{')[0]
            for fname in ('japan_open_world-linux-x86_64.tar.gz', 'SHA256SUMS.txt', 'build_report.json'):
                fp = os.path.join(build_dir, fname)
                url = f'{upload}?name={fname}'
                r = _req.Request(url, data=open(fp, 'rb').read(), method='POST',
                                 headers={**headers, 'Content-Type': 'application/octet-stream'})
                try:
                    with _req.urlopen(r, timeout=600) as resp:
                        print('Uploaded:', fname, resp.status)
                except Exception as exc:
                    print('Upload failed:', fname, exc)
            release_url = rel.get('html_url')
            print('Release:', release_url)
        else:
            print('Release creation failed (tag may already exist, or token lacks scope).')
else:
    print('Release publishing not requested.')


In [ ]:
# ---------------------------------------------------------------------------
# 6. Final summary
# ---------------------------------------------------------------------------
print('=' * 68)
print(' JAPAN OPEN WORLD - BUILD SUMMARY')
print('=' * 68)
print(f'Repository : {REPO_URL}')
print(f'Commit     : {commit_sha}')
print(f'Build dir  : {build_dir}')
print(f'Executable : {os.path.join(build_dir, "japan_open_world.x86_64")}')
print(f'Archive    : {os.path.join(build_dir, "japan_open_world-linux-x86_64.tar.gz")}')
print(f'Checksums  : {os.path.join(build_dir, "SHA256SUMS.txt")}')
print(f'Report     : {os.path.join(build_dir, "build_report.json")}')
if drive_path: print(f'Drive copy : {drive_path}')
if release_url: print(f'Release    : {release_url}')

print('\nTo play locally on Linux:')
print(f'  tar -xzf japan_open_world-linux-x86_64.tar.gz && ./japan_open_world.x86_64')
print('\nControls: WASD move, Shift sprint, Space jump, E interact, C camera, Esc pause')


## Notes and limitations

- The **build** is headless: Colab has no GPU-backed display during compilation, so
  compilation, resource integrity, world streaming, physics settling and packaged
  startup are verified there.
- The **visual acceptance stage** (cell 3b) is *not* headless. It provisions a display,
  runs the packaged game with a real graphics backend (`vulkan` when the runtime exposes
  a Vulkan ICD, otherwise Mesa llvmpipe `opengl3`), captures five framed views of the
  streamed world plus a gameplay video, and verifies the pixels independently. It fails
  loudly when the runtime cannot produce a graphics context.
- On a Tesla T4 with a working Vulkan ICD the capture runs on the GPU in Forward+.
  Without one it runs on Mesa llvmpipe: correct output, but only a few frames per second,
  which is why the video length is a runtime parameter rather than a fixed 30 s.
- The world covers a **~4 km x 4 km slice of central Kyoto** built from real elevation and
  OpenStreetMap data, with real MLIT PLATEAU LOD2 building geometry. It is not a
  country-scale world, and building interiors are not modelled.
- Geometry is streamed around the player; there is no cache-busting rebuild of untouched
  chunks, which keeps reruns fast.

- PLATEAU 3D city models: `tools/fetch_plateau.py` converts official MLIT PLATEAU Kyoto
  building geometry (Draco-compressed 3D Tiles) into Godot .pmesh files. The converted
  .pmesh files are gitignored; `export_presets.cfg` includes them in the PCK via the
  `*.pmesh` include filter, without which the packaged game silently ships no PLATEAU
  geometry and falls back to OSM boxes.
